In [ ]:
!pip install transformers datasets librosa scikit-learn

In [ ]:
!pip install --upgrade transformers datasets evaluate


In [ ]:
pip install torch==2.7.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.0%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.2.0%2Bcu118-cp311-cp311-linux_x86_64.whl (3.3 MB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cuda_nvrtc_cu11-11.8.89-py3-none-manylinux1_x86_64.whl (23.2 MB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cuda_runtime_cu11-11.8.89-py3-none-manylinux1_x86_64.whl (875 kB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cuda_cupti_cu11-11.8.87-py3-none-manylinux1_x86_64.whl (13.1 MB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cudnn_cu11-9.1.0.70-py3-none-manylinux2014_x86_64.whl (663.9 MB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cublas_cu11-11.11.3.6-py3-none-man

In [ ]:
!pip install pyctcdecode

In [ ]:
!pip show torch

Name: torch
Version: 2.6.0+cu124
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision


In [ ]:
import os
import pandas as pd
import torchaudio
import torch
from torch.utils.data import Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, TrainingArguments, Trainer
import evaluate


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("/content/drive/MyDrive/SER-Model/dataset.csv")

# Replace backslashes with forward slashes
df["path"] = df["path"].str.replace("\\", "/", regex=False)

# Tambahkan full path
df["path"] = "/content/drive/MyDrive/SER-Model/" + df["path"]

# Mapping label ke ID
label_list = sorted(df["emotion"].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# Tambahkan kolom label numerik
df["label"] = df["emotion"].map(label2id)


In [ ]:
# Pretrained processor

class SERDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        speech_array, sampling_rate = torchaudio.load(row["path"])
        speech_array = speech_array.squeeze()  # mono
        # Resample to 16kHz if needed
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
            speech_array = resampler(speech_array)

        inputs = processor(speech_array, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs["labels"] = torch.tensor(label2id[row["emotion"]])
        return {k: v.squeeze() for k, v in inputs.items()}

In [ ]:
# Split train/test
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
train_dataset = SERDataset(train_df)
val_dataset = SERDataset(val_df)

# Load model
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor
from transformers import AutoProcessor, AutoModelForCTC
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "r-f/wav2vec-english-speech-emotion-recognition",
    num_labels=len(label_list),     # overwrite sesuai label dataset kamu
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True    # penting jika jumlah label berbeda dengan model aslinya
)

processor = Wav2Vec2FeatureExtractor.from_pretrained("r-f/wav2vec-english-speech-emotion-recognition")


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at r-f/wav2vec-english-speech-emotion-recognition and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Evaluation metric

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=-1)
    return accuracy.compute(predictions=preds, references=labels)

In [ ]:
# Training
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=10,
    per_device_eval_batch_size=5,
    num_train_epochs=15,
    learning_rate=1e-3,
    warmup_steps=50,
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to= "none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-15-906409906.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.491800,1.359069,0.367241
2,1.089900,0.901234,0.567241
3,0.952300,1.243028,0.486207
4,0.806200,0.642257,0.775862
5,0.559100,0.516084,0.827586
6,0.453800,0.636887,0.806897
7,0.352700,0.252633,0.920690
8,0.383300,0.222649,0.932759
9,0.253200,0.195463,0.946552
10,0.129600,0.185988,0.948276


TrainOutput(global_step=3480, training_loss=0.5016867278360537, metrics={'train_runtime': 4811.9293, 'train_samples_per_second': 7.232, 'train_steps_per_second': 0.723, 'total_flos': 3.1117134518831334e+18, 'train_loss': 0.5016867278360537, 'epoch': 15.0})

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Dapatkan prediksi dari trainer
predictions = trainer.predict(val_dataset)

# Ambil prediksi logits dan ubah ke label prediksi
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Tampilkan classification report
print(classification_report(y_true, y_pred, target_names=label_list))


              precision    recall  f1-score   support

       angry       0.99      1.00      0.99        80
     disgust       0.94      0.97      0.95        92
        fear       1.00      1.00      1.00        80
       happy       0.99      0.97      0.98        92
     neutral       1.00      0.99      0.99        92
         sad       0.97      0.97      0.97        92
   surprised       0.96      0.94      0.95        52

    accuracy                           0.98       580
   macro avg       0.98      0.98      0.98       580
weighted avg       0.98      0.98      0.98       580



In [ ]:
!pip install sounddevice scipy

In [ ]:
from google.colab import files

# Upload file audio (misal hasil rekaman dari HP/PC)
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving Happy.m4a to Happy.m4a


In [ ]:
import torchaudio
import torch

def predict_emotion_from_audio(file_path, model, processor, id2label, device="cuda" if torch.cuda.is_available() else "cpu"):
    model = model.to(device)
    model.eval()

    speech_array, sampling_rate = torchaudio.load(file_path)

    # Pastikan mono (1D tensor)
    if speech_array.shape[0] > 1:
        speech_array = speech_array.mean(dim=0)  # konversi stereo ke mono

    speech_array = speech_array.squeeze()

    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
        speech_array = resampler(speech_array)

    inputs = processor(speech_array, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits
        predicted_id = torch.argmax(logits, dim=-1).item()

    return id2label[predicted_id]

# Inference
predicted_emotion = predict_emotion_from_audio(file_path, model, processor, id2label)
print(f"🧠 Predicted Emotion: {predicted_emotion}")


🧠 Predicted Emotion: sad


In [ ]:
!pip install huggingface_hub



In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import Trainer
from huggingface_hub import HfApi

username = "FadQ"   # misal: fadhilqashmal
repo_name = "wav2vec2-ser-FineTune-TESS-IWS"      # bebas
full_repo_name = f"{username}/{repo_name}"

# Buat repo baru
api = HfApi()
api.create_repo(repo_id=repo_name, exist_ok=True)
# Simpan model ke direktori lokal dulu
trainer.save_model("ser-finetuned")               # model
processor.save_pretrained("ser-FineTune-TESS-IWS")        # processor

# Push ke Hugging Face Hub
trainer.push_to_hub()
processor.push_to_hub(repo_id=full_repo_name)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/FadQ/wav2vec2-ser-FineTune-TESS-IWS/commit/62d9675a9f6a6e5a4e09123c41456120a1c43ad4', commit_message='Upload feature extractor', commit_description='', oid='62d9675a9f6a6e5a4e09123c41456120a1c43ad4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FadQ/wav2vec2-ser-FineTune-TESS-IWS', endpoint='https://huggingface.co', repo_type='model', repo_id='FadQ/wav2vec2-ser-FineTune-TESS-IWS'), pr_revision=None, pr_num=None)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

def compute_metrics(pred):
    logits, labels = pred.predictions, pred.label_ids
    preds = np.argmax(logits, axis=1)

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

    # Tambahkan AUC ROC hanya jika jumlah kelas == 2 atau multilabel support
    try:
        # Untuk multiclass AUC, gunakan 'ovr' (one-vs-rest)
        auc = roc_auc_score(labels, logits, multi_class="ovr")
        metrics["roc_auc"] = auc
    except ValueError:
        # roc_auc tidak dihitung jika tidak valid
        metrics["roc_auc"] = None

    return metrics


In [ ]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.10608037561178207, 'eval_accuracy': 0.9775862068965517, 'eval_runtime': 10.1668, 'eval_samples_per_second': 57.048, 'eval_steps_per_second': 11.41, 'epoch': 15.0}
